# Select a research loss and train a manifold projection

This synthetic tutorial checks API behavior, not NLP benchmark performance.
Install from the repository: `python -m pip install -e ".[train,viz]"`.
Change `loss_name` below to compare objectives with the same split and seed.
Use validation data to choose a loss; do not tune against the test accuracy here.


In [ ]:
import numpy as np
import torch
from manifold_studio import Sphere, Torus
from manifold_studio.training import TrainableProjector
from manifold_studio.losses import available_losses, make_loss, fixed_spectral_basis

torch.manual_seed(7)
torch.set_num_threads(1)
geometry = Sphere() * Torus()
loss_name = "smtl"  # Try geodesic_triplet, euclidean_triplet, distance_alignment...
metric = "intrinsic"  # Use ambient for Möbius or either ambient triplet name.
if loss_name in ("euclidean_triplet", "ambient_triplet"):
    metric = "ambient"
available_losses()


In [ ]:
# Split before normalization, graph construction or learning.
centers = torch.randn(3, 12)
train_y = torch.arange(3).repeat_interleave(16)
test_y = torch.arange(3).repeat_interleave(5)
X_train = centers[train_y] + 0.5 * torch.randn(48, 12)
X_test = centers[test_y] + 0.5 * torch.randn(15, 12)
q = centers[:1]
triplets = torch.tensor([[i, (i//16)*16+(i+1)%16, (i+16)%48] for i in range(48)])
U = fixed_spectral_basis(X_train)
model = TrainableProjector(12, geometry).fit_normalization(X_train)
objective = make_loss(loss_name, geometry, metric=metric)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)


In [ ]:
history = []
for step in range(80):
    optimizer.zero_grad()
    result = objective(model(X_train), source=X_train, triplets=triplets,
                       query_parameters=model(q), basis=U)
    result.total.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
    optimizer.step()
    history.append(result.detached())
model.eval().fit_display(X_train)
train_embedding = model.transform(X_train)
test_embedding = model.transform(X_test)
# Simple fixed 1-NN probe: fit/graph/normalization never see test rows.
D = geometry.distance(test_embedding.parameters, train_embedding.parameters, metric=metric)
pred = train_y.numpy()[D.argmin(axis=1)]
source_D = torch.cdist(X_test, X_train)
source_pred = train_y[source_D.argmin(dim=1)]
print({"synthetic_source_1nn": float((source_pred == test_y).float().mean()),
       "synthetic_trained_1nn": float(np.mean(pred == test_y.numpy())),
       "initial_loss": history[0]["total"], "last_pre_update_loss": history[-1]["total"]})


In [ ]:
from pathlib import Path
out = Path("outputs/loss_tutorial")
out.mkdir(parents=True, exist_ok=True)
model.save(out / "trained_projector.npz")
test_embedding.save(out / "heldout_embeddings.npz")
restored = TrainableProjector.load(out / "trained_projector.npz")
np.testing.assert_allclose(restored.transform(X_test).coordinates, test_embedding.coordinates)
train_embedding.plot_3d(labels=train_y.numpy(), target_index=0).write_html(out / "trained.html")
print(out.resolve())


## Interpret the result

SMTL and spectral components are experimental. Loss reduction or concentrated
spectral energy can coexist with worse retrieval. This toy split is not evidence
for your paper. Compare source embeddings, a same-dimension PCA baseline and
trained geometries on a real held-out task with multiple seeds. See `docs/LOSSES.md`
for formulas, fixed-graph limitations, historical sources and excluded losses.
